In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, f1_score
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer 
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('TelcoChurn.csv')
df.head()

In [ ]:
df.info()

In [ ]:
for col in df:
    print(f"{col}: {df[col].value_counts()} unique values")
    print(10*"-----")

In [ ]:
# TotalCharges düzeltme
df['TotalCharges'] = df['TotalCharges'].replace(' ', 0).astype(float)

# customerID drop
df.drop('customerID', axis=1, inplace=True)

# Feature engineering - Basic
df['IsMonthToMonth'] = np.where(df['Contract'] == 'Month-to-month', 1, 0)
df['IsAutoPay'] = np.where(df['PaymentMethod'].str.contains('automatic'), 1, 0)
df['HasPhoneService'] = np.where(df['PhoneService'] == 'Yes', 1, 0)
df['HasInternetService'] = np.where(df['InternetService'] == 'No', 0, 1)
df['IsAlone'] = np.where((df['Partner'] == 'No') & (df['Dependents'] == 'No'), 1, 0)

In [ ]:
# Feature engineering - Financial
df['AvgMonthlyCharges'] = df['TotalCharges'] / (df['tenure'] + 1)
df['PriceIncrease'] = df['MonthlyCharges'] / (df['AvgMonthlyCharges'] + 0.01)
df['ExpectedRevenue'] = df['MonthlyCharges'] * 12

# Feature engineering - Service Combinations
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                'TechSupport', 'StreamingTV', 'StreamingMovies']
df['TotalServices'] = sum((df[col] == 'Yes').astype(int) for col in service_cols)
df['IsPremiumCustomer'] = (df['TotalServices'] >= 4).astype(int)
df['NoExtraServices'] = (df['TotalServices'] == 0).astype(int)

# Feature engineering - Risk Factors
df['HighRiskProfile'] = ((df['Contract'] == 'Month-to-month') & 
                          (df['PaymentMethod'] == 'Electronic check')).astype(int)
df['IsNewCustomer'] = (df['tenure'] <= 6).astype(int)
df['LoyaltyScore'] = df['tenure'] * (3 - df['IsMonthToMonth'] * 2)
df['FirstYear'] = (df['tenure'] <= 12).astype(int)

# Feature engineering - Internet Patterns
df['FiberNoSecurity'] = ((df['InternetService'] == 'Fiber optic') & 
                          (df['OnlineSecurity'] == 'No')).astype(int)

# Feature engineering - Demographics
df['SeniorAlone'] = ((df['SeniorCitizen'] == 1) & (df['IsAlone'] == 1)).astype(int)
df['YoungFamily'] = ((df['SeniorCitizen'] == 0) & 
                      (df['Partner'] == 'Yes') & 
                      (df['Dependents'] == 'Yes')).astype(int)


In [ ]:
df.info()

In [ ]:
#Find Outliers with IQR
def find_outliers_iqr(df, threshold = 1.5):
    outlier_summary = {}


    numeric_cols = df.select_dtypes(include=["float64", "int64"]).columns
   
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1


        lower_bound = Q1 - threshold * IQR
        upper_bound = Q3 + threshold * IQR


        outliers = df[ (df[col] < lower_bound) | (df[col] > upper_bound)]
       
        outlier_summary[col] = {
            "outlier_count" : outliers.shape[0],
            "outlier_percentage" : 100 * outliers.shape[0] / df.shape[0],
            "lower_bound" : lower_bound,
            "upper_bound" : upper_bound
        }
    return pd.DataFrame(outlier_summary)
outlier_info = find_outliers_iqr(df)
print(outlier_info)

In [ ]:
#Pairplot
sns.pairplot(df, hue='Churn')

In [ ]:
# Create histograms for all columns
num_cols = len(df.columns)
nrows = (num_cols + 4) // 5  # Calculate rows needed for 5 columns per row
fig, axes = plt.subplots(nrows=nrows, ncols=5, figsize=(20, nrows*3))
axes = axes.ravel()

for i, col in enumerate(df.columns):
    if df[col].dtype in ['int64', 'float64']:
        df[col].hist(ax=axes[i])
    else:
        df[col].value_counts().plot(kind='bar', ax=axes[i])
    
    axes[i].set_title(col)
    axes[i].tick_params(axis='x', rotation=45)

# Hide any unused subplots
for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
df.isnull().sum()

In [ ]:
#Columns 
binary_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
               'PaperlessBilling', 'IsMonthToMonth', 'IsAutoPay', 'HasPhoneService',
               'HasInternetService', 'IsAlone', 'IsPremiumCustomer', 'NoExtraServices',
               'HighRiskProfile', 'IsNewCustomer', 'FirstYear', 'FiberNoSecurity',
               'SeniorAlone', 'YoungFamily']
cat_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
            'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
            'Contract', 'PaymentMethod']
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlyCharges', 
            'PriceIncrease', 'ExpectedRevenue', 'TotalServices', 'LoyaltyScore']

In [ ]:
# Pipelines
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

binary_cols_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='if_binary', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols),
    ('bin', binary_cols_pipeline, binary_cols)
], remainder='drop')

In [ ]:
#Class imbalance handling
X = df.drop('Churn', axis=1)
y = df['Churn'].map({'No': 0, 'Yes': 1})
class_ratio = (y == 0).sum() / (y == 1).sum()
print(f"Class imbalance ratio: {class_ratio:.2f}")
print(f"Churn distribution:\n{y.value_counts()}")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# RandomForest parameter grid
rf_params = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [3, 5, 7, 9, 11, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False]
}
pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model',RandomForestClassifier())
])

In [ ]:
# RandomizedSearchCV
rf_search = RandomizedSearchCV(
    pipeline_rf, 
    {'model__' + k: v for k, v in rf_params.items()},
    n_iter=20,
    scoring='f1',
    cv=5,
    verbose=1,
    n_jobs=-1,
    random_state=42
)
rf_search.fit(X_train, y_train)
y_pred = rf_search.predict(X_test)

In [ ]:
print("\nResults:")
print(f"Best params: {rf_search.best_params_}")
print(classification_report(y_test, y_pred))
print(f"ROC AUC Score: {roc_auc_score(y_test, y_pred):.4f}")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")

In [ ]:
#Roc Curve
from sklearn.metrics import roc_curve, precision_recall_curve
plt.figure(figsize=(10,6))
roc = roc_auc_score(y_test, rf_search.predict_proba(X_test)[:, 1])
fpr, tpr, _ = roc_curve(y_test, rf_search.predict_proba(X_test)[:, 1])
plt.plot(fpr, tpr, label=f'Random Forest (AUC = {roc:.4f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

In [ ]:
#RecalL Curve
y_proba = rf_search.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
plt.figure(figsize=(10,6))
plt.plot(recall, precision, label='Random Forest')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.show()

In [ ]:
# Feature importance
print("\n" + "="*50)
print("Top 15 Most Important Features")
print("="*50)
feature_names = rf_search.best_estimator_.named_steps['preprocessor'].get_feature_names_out()
feature_importance = rf_search.best_estimator_.named_steps['model'].feature_importances_
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False).head(30)
print(importance_df)

In [ ]:
import pickle
with open('rf_telco_churn_model.pkl', 'wb') as f:
    pickle.dump(pipeline_rf, f)